# Random forests for regression

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
A single tree is unstable, but **the average of hundreds of different trees** is stable and accurate. It is like asking a large crowd instead of one person and taking the average (the **wisdom of crowds**).

## Two sources of randomness
- **Bagging:** each tree is trained on a **bootstrap sample**: n samples drawn with replacement. About 63% of the samples land in each bootstrap; the rest (**out-of-bag**, OOB) give a free evaluation.
- **Random feature choice:** at each node, only a random subset of the features is considered (`max_features`). This makes the trees **more independent**.

$$\hat y(x)=\frac1B\sum_{b=1}^{B}T_b(x)$$

## Why does averaging help?
For $B$ predictions with variance $\sigma^2$ and correlation $\rho$, the variance of their mean is:

$$\text{Var}=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$$

More trees ($B$) remove the second term; lower correlation $\rho$ (random features) reduces the first.

> [1 tree: jumpy and noisy. 10 trees: better. 200 trees: smooth and stable.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/forest_reg_n.png)

*1 tree: jumpy and noisy. 10 trees: better. 200 trees: smooth and stable.*

[[lab:reg_forest]]

In the lab, the faint lines are single trees and the bold line is their average. Raise the number of trees from 1.

## Hyperparameters
| Hyperparameter | Default | Meaning |
|---|---|---|
| `n_estimators` | 100 | Number of trees; more is better but slower (it does not cause overfitting) |
| `max_features` | 1.0 for regression | Share of features per node; try 0.3 to 0.5 |
| `max_depth` / `min_samples_leaf` | none / 1 | Complexity of each tree |
| `oob_score` | False | Free evaluation on the out-of-bag samples |



In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

X, y = load_diabetes(return_X_y=True)
for n in [1, 10, 50, 200]:
    rf = RandomForestRegressor(n_estimators=n, min_samples_leaf=5, max_features=0.5, oob_score=n > 20, random_state=0)
    cv = cross_val_score(rf, X, y, cv=5, scoring="r2").mean()
    rf.fit(X, y)
    oob = f"{rf.oob_score_:.3f}" if n > 20 else "-"
    print(f"{n:4d} trees: CV R2 = {cv:.3f}   OOB R2 = {oob}")

# Share of unique samples in one bootstrap sample
rng = np.random.default_rng(0)
print("unique share:", len(np.unique(rng.integers(0, 1000, 1000))) / 1000)



## Feature importance
The forest reports how much each feature reduced the error. This measure is biased toward features with many unique values; **permutation importance** (`permutation_importance`) is more reliable:



In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

d = load_diabetes(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(d.data, d.target, random_state=0)
rf = RandomForestRegressor(300, min_samples_leaf=5, random_state=0).fit(X_tr, y_tr)
imp = pd.DataFrame({"impurity": rf.feature_importances_,
                    "permutation": permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0).importances_mean},
                   index=d.data.columns)
print(imp.sort_values("permutation", ascending=False).round(3))



| Pros | Cons |
|---|---|
| Accurate with little tuning | Harder to interpret than one tree |
| Resistant to overfitting | Larger, slower to predict |
| Needs no scaling | Does not extrapolate |
| Free OOB evaluation | |

### Exercise

Try max_features 0.2, 0.5 and 1.0 and compare the CV R².

In [ ]:
# Your answer here